# Capybara RC3 — Sphinx recursion hotfix

Run the cell below from the root of the existing local Git clone. It repairs the recursive mirrored-document include, patches the RC3 finalizer, commits, and pushes `main`.


In [1]:
# CAPYBARA RC3 — HOTFIX: MyST/Sphinx recursive mirrored-document include
#
# Run this from the ROOT of your local Git clone:
#   BrianBowers-NapaCounty/itam-itsm_integration_framework
#
# This:
#   1) restores CONTRIBUTING.md as a real canonical document,
#   2) materializes docs/contributing.md, docs/changelog.md, and
#      docs/code-of-conduct.md as direct content instead of recursive includes,
#   3) patches tools/finalize_rc3.py so rerunning finalization cannot recreate
#      the recursion,
#   4) removes any stale generated *.md.rst wrappers if they exist,
#   5) commits and pushes the hotfix to origin/main.
#
# Do NOT "fix" this by raising sys.setrecursionlimit(). The recursion is a
# source-structure bug, not a legitimately deep document.

from pathlib import Path
import subprocess, re, sys, importlib.util

ROOT = Path.cwd().resolve()

required = [
    ROOT / ".git",
    ROOT / ".readthedocs.yaml",
    ROOT / "docs",
    ROOT / "tools" / "finalize_rc3.py",
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise RuntimeError(
        "Run this cell from the LOCAL GIT REPOSITORY ROOT.\nMissing:\n  "
        + "\n  ".join(missing)
    )

def git(*args, check=True):
    r = subprocess.run(
        ["git", "-C", str(ROOT), *args],
        capture_output=True,
        text=True
    )
    if r.stdout.strip():
        print(r.stdout.strip())
    if r.stderr.strip():
        print(r.stderr.strip())
    if check and r.returncode != 0:
        raise RuntimeError(
            f"git {' '.join(args)} failed ({r.returncode})\n{r.stderr}"
        )
    return r

print("Repository:", ROOT)
print("Branch:", git("branch", "--show-current").stdout.strip())

CONTRIBUTING = r'''# Contributing to Capybara

Thank you for your interest in contributing to Capybara. Community contributions are welcome, encouraged, and essential to the long-term health of this project.

This document explains how to contribute in a way that is consistent with the project's goals, structure, and licensing.

---

## License Context

This repository is licensed under the MIT License, as described in the `LICENSE` file.

By submitting a contribution, you agree that:

- Your contributions are provided freely and openly to the global community.
- Your contributions may be used, modified, merged, published, distributed, sublicensed, and/or sold, consistent with the MIT License.
- You grant the project maintainers and all downstream users the same permissions granted by the MIT License, without additional restrictions.

### Attribution Requirement

While the MIT License permits broad reuse, this project includes the following project-level requirement, which contributors must respect:

- The file `AUTHORS.md` must not be modified.
- The file `AUTHORS.md` must be included in all future forks, redistributions, and derivative works of this repository.

This requirement exists to preserve authorship history and credit. Contributions that attempt to remove, alter, or bypass this requirement will not be accepted.

---

## How to Contribute

You may contribute in many ways, including but not limited to:

- Documentation improvements
- New content, examples, or frameworks
- Bug fixes or corrections
- Structural or organizational improvements
- Clarifications or refinements to existing material

All contributions should be made via pull requests and should follow the conventions described below.

---

## Repository Conventions

To keep the repository organized, maintainable, and navigable, contributors are expected to follow these conventions.

### Folder Structure

- Whenever possible, new content should be added to an appropriate subfolder, whether existing or newly created.
- Adding content directly to the top-level `Capybara` folder is strongly discouraged, except where explicitly required by existing structure or maintainers.
- If a suitable subfolder does not exist, contributors are encouraged to create one.

### README Files

- All new subfolders must include a `README.md` describing their purpose, contents, and intended usage.
- For existing subfolders, contributors should update and maintain the associated `README.md` when changes are made.

Clear, accurate documentation is considered a core part of any contribution.

### Changelog Updates

- The file `CHANGELOG.md` in the top-level `Capybara` folder must always be updated to reflect any additions, removals, or modifications made to the repository.
- Changelog entries should be concise, descriptive, and grouped under the appropriate version heading.

Pull requests that modify content without a corresponding changelog update may be requested to revise before acceptance.

---

## Versioning and Releases

This project follows Semantic Versioning (SemVer):

`MAJOR.MINOR.PATCH`

Each version must be preceded by an Edition Codename, chosen by the contributor or release author.

Examples:

- `Capybara: Purple Edition 3.5.14`
- `Capybara: Verdant Edition 2.1.0`

### Initial Release

The initial public release, planned for September 2026, will be:

- **Capybara: Original Edition 1.0.0**

Subsequent releases should increment version numbers according to SemVer rules and include meaningful changelog entries.

---

## Publishing a New Edition and Version

Capybara editions and versions are published using standard Git and GitHub workflows, following Semantic Versioning (SemVer) and the project's Edition + Version naming convention.

The **Publishing a New Edition** section explains how to publish a new edition using either the command-line Git workflow or the GitHub web interface, using the transition from:

> Capybara: Alpha Edition 0.x.y to Capybara: Original Edition 1.0.0

as a real-world example.

### After Publishing

Once a new edition is published:

- The release tag (`v1.0.0`) becomes the canonical reference point for that edition.
- Documentation hosting platforms (such as Read the Docs) may be configured to expose the version publicly.
- All future development should proceed on `main` toward the next edition or version (for example, `1.1.0` or a future edition).

### Notes on Pre-Releases

Pre-release editions (such as Alpha or Beta editions using `0.x.y` versions):

- May be published without tags.
- Should clearly indicate their pre-release status in documentation.
- Do not imply long-term stability or backward compatibility.

The Original Edition 1.0.0 represents Capybara's first stable, production-ready release and establishes the baseline for all future versions.

By following these steps, contributors help ensure Capybara releases remain consistent, traceable, and easy for the global community to adopt and build upon.

---

## Style Notes and Guidelines

The **Style Notes and Guidelines** section in the project's guide back-matter establishes the baseline stylistic and structural conventions for this repository.

- All contributors are expected to honor the examples and guidelines in that section.
- Deviations from established style patterns are discouraged, but not forbidden.

### Intentional Style Changes

If a contributor makes a carefully considered decision to depart from existing style precedents:

- The new version's **Style Notes and Guidelines** section must be updated to document and justify the new style choices.
- The change should be explained clearly in the changelog.

This ensures that style evolution is intentional, documented, and understandable to future contributors.

---

## Original Edition 1.0.0 Contribution Requirements

Contributions should preserve authoritative-system boundaries, platform-neutral core logic, U.S. English spelling, accessible documentation, and dry-run-first examples.

Open an issue or pull request describing the use case, affected platforms, data ownership, and test approach.

---

## Final Notes

- Contributions should be respectful, constructive, and well-documented.
- Large or structural changes are encouraged to be discussed in an issue before submission.
- Maintainers reserve the right to request revisions to ensure consistency with this guide.

Thank you for helping make Capybara better for everyone.

---
'''

(ROOT / "CONTRIBUTING.md").write_text(CONTRIBUTING, encoding="utf-8")

def write_mirror(root_name, docs_name):
    src = ROOT / root_name
    dst = ROOT / "docs" / docs_name
    if not src.exists():
        raise RuntimeError(f"Canonical root file missing: {src}")
    canonical = src.read_text(encoding="utf-8", errors="replace").strip()
    note = (
        f"This page mirrors the repository's `/{root_name}`. "
        "The documentation copy is materialized during release finalization "
        "to avoid recursive include behavior in MyST/Sphinx.\n\n"
    )
    dst.write_text(note + canonical + "\n", encoding="utf-8")
    print("Materialized:", dst.relative_to(ROOT))

write_mirror("CONTRIBUTING.md", "contributing.md")
write_mirror("CHANGELOG.md", "changelog.md")
write_mirror("CODE_OF_CONDUCT.md", "code-of-conduct.md")

stale = [
    ROOT / "CONTRIBUTING.md.rst",
    ROOT / "docs" / "contributing.md.rst",
    ROOT / "docs" / "changelog.md.rst",
    ROOT / "docs" / "code-of-conduct.md.rst",
]
for p in stale:
    if p.exists():
        p.unlink()
        print("Removed stale wrapper:", p.relative_to(ROOT))

finalizer = ROOT / "tools" / "finalize_rc3.py"
s = finalizer.read_text(encoding="utf-8")

dangerous = '''    if (docs / "contributing.md").exists():
        shutil.copy2(docs / "contributing.md", repo / "CONTRIBUTING.md")
'''

replacement = '''    # Canonical mirrored repository documents flow ROOT -> docs only.
    # Never copy docs/contributing.md back over root CONTRIBUTING.md:
    # the historical docs page may contain a MyST include of the root file,
    # which would make the root file recursively include itself.
    for root_name, docs_name in [
        ("CONTRIBUTING.md", "contributing.md"),
        ("CHANGELOG.md", "changelog.md"),
        ("CODE_OF_CONDUCT.md", "code-of-conduct.md"),
    ]:
        source = repo / root_name
        destination = docs / docs_name
        if source.exists():
            canonical = source.read_text(encoding="utf-8", errors="replace").strip()
            note = (
                f"This page mirrors the repository's `/{root_name}`. "
                "The documentation copy is materialized during release finalization "
                "to avoid recursive include behavior in MyST/Sphinx.\\n\\n"
            )
            destination.write_text(note + canonical + "\\n", encoding="utf-8")
'''

if dangerous in s:
    s = s.replace(dangerous, replacement)
elif "Never copy docs/contributing.md back over root CONTRIBUTING.md" not in s:
    raise RuntimeError(
        "Could not identify the expected RC3 finalizer block. "
        "Stop here rather than patching an unknown revision."
    )

finalizer.write_text(s, encoding="utf-8")
print("Patched:", finalizer.relative_to(ROOT))

bad = []
for rel in [
    "CONTRIBUTING.md",
    "docs/contributing.md",
    "docs/changelog.md",
    "docs/code-of-conduct.md",
]:
    t = (ROOT / rel).read_text(encoding="utf-8", errors="replace")
    if "```{include}" in t or "{include}" in t or ".. include::" in t:
        bad.append(rel)

if bad:
    raise RuntimeError(
        "Recursive/include wrappers still present in: " + ", ".join(bad)
    )

compile(finalizer.read_text(encoding="utf-8"), str(finalizer), "exec")

print("\n✓ Mirrored-document recursion removed.")
print("✓ Root CONTRIBUTING.md restored as direct content.")
print("✓ RC3 finalizer patched against recurrence.")

if importlib.util.find_spec("sphinx") is not None:
    print("\nRunning local Sphinx HTML smoke test...")
    build_dir = ROOT / "_build_hotfix_html"
    r = subprocess.run(
        [
            sys.executable, "-m", "sphinx",
            "-T", "-b", "html",
            str(ROOT / "docs"),
            str(build_dir),
        ],
        text=True
    )
    if r.returncode != 0:
        raise RuntimeError(
            "Local Sphinx smoke test failed. Do NOT push yet; inspect output above."
        )
    print("✓ Local Sphinx HTML build completed.")
else:
    print("\nSphinx is not installed in this kernel; skipping local smoke test.")

git("add", "-A")

status = git("status", "--porcelain", check=False).stdout.strip()
if not status:
    print("\nNo Git changes remain to commit.")
else:
    git(
        "commit",
        "-m",
        "Fix recursive MyST mirror includes in RC3 documentation",
    )

confirm = input(
    "\nType PUSH HOTFIX to push this repair to origin/main: "
).strip()

if confirm != "PUSH HOTFIX":
    raise RuntimeError("Push cancelled.")

git("push", "origin", "main")

print("\n" + "=" * 72)
print("RC3 SPHINX RECURSION HOTFIX PUSHED")
print("=" * 72)
print("Read the Docs should rebuild from the new main commit via its webhook.")
print("If it does not start automatically, trigger a new 'latest' build in RTD.")
print("=" * 72)


Repository: C:\Users\BBOWERS\Jupyter Notebooks\Capybara_Framework_GitHub_Repository_RC3_Integrated
main
Branch: main
Materialized: docs\contributing.md
Materialized: docs\changelog.md
Materialized: docs\code-of-conduct.md
Patched: tools\finalize_rc3.py

✓ Mirrored-document recursion removed.
✓ Root CONTRIBUTING.md restored as direct content.
✓ RC3 finalizer patched against recurrence.

Sphinx is not installed in this kernel; skipping local smoke test.
M  CONTRIBUTING.md
M  Capybara_RC3_Integrated_Deployment.ipynb
A  Capybara_RC3_Sphinx_Recursion_Hotfix.ipynb
M  docs/changelog.md
M  docs/code-of-conduct.md
M  docs/contributing.md
M  tools/finalize_rc3.py
[main fcd2316] Fix recursive MyST mirror includes in RC3 documentation
 7 files changed, 1306 insertions(+), 40 deletions(-)
 create mode 100644 Capybara_RC3_Sphinx_Recursion_Hotfix.ipynb



Type PUSH HOTFIX to push this repair to origin/main:  PUSH HOTFIX


To https://github.com/BrianBowers-NapaCounty/itam-itsm_integration_framework.git
   c2a0a4f..fcd2316  main -> main

RC3 SPHINX RECURSION HOTFIX PUSHED
Read the Docs should rebuild from the new main commit via its webhook.
If it does not start automatically, trigger a new 'latest' build in RTD.


In [2]:
# =====================================================================
# CAPYBARA — TRIGGER + MONITOR READ THE DOCS BUILD
# =====================================================================

import requests
import getpass
import time
import json
import re

RTD_PROJECT_SLUG = "capybara-framework"
RTD_VERSION = "latest"
RTD_API_BASE = "https://app.readthedocs.org/api/v3"

# Reuse an existing token from this kernel if available.
RTD_API_TOKEN = globals().get("RTD_API_TOKEN", "").strip()

if not RTD_API_TOKEN:
    RTD_API_TOKEN = getpass.getpass(
        "Read the Docs API token (hidden): "
    ).strip()
    globals()["RTD_API_TOKEN"] = RTD_API_TOKEN

if not RTD_API_TOKEN:
    raise RuntimeError("No Read the Docs API token supplied.")


def rtd(method, path, body=None, allowed=(200, 201, 202, 204)):
    url = (
        path
        if path.startswith("http")
        else RTD_API_BASE.rstrip("/") + "/" + path.lstrip("/")
    )

    headers = {
        "Accept": "application/json",
        "Authorization": f"Token {RTD_API_TOKEN}",
        "User-Agent": "Capybara-Framework-Build-Trigger/1.0",
    }

    if body is not None:
        headers["Content-Type"] = "application/json"

    response = requests.request(
        method.upper(),
        url,
        headers=headers,
        json=body,
        timeout=60,
    )

    try:
        data = response.json() if response.text else None
    except Exception:
        data = response.text

    if response.status_code not in allowed:
        raise RuntimeError({
            "method": method,
            "status": response.status_code,
            "url": url,
            "response": data,
        })

    return response.status_code, data


def build_state(build):
    state = build.get("state")

    if isinstance(state, dict):
        return (
            state.get("code")
            or state.get("name")
            or str(state)
        )

    return str(state or "")


def get_build_id(payload):
    if not isinstance(payload, dict):
        return None

    build = payload.get("build")

    if isinstance(build, int):
        return build

    if isinstance(build, dict):
        if build.get("id"):
            return int(build["id"])

        for key in ("url", "_links"):
            value = build.get(key)

            if isinstance(value, str):
                m = re.search(r"/builds/(\d+)/?", value)
                if m:
                    return int(m.group(1))

            elif isinstance(value, dict):
                for link in value.values():
                    if isinstance(link, str):
                        m = re.search(r"/builds/(\d+)/?", link)
                        if m:
                            return int(m.group(1))

    if isinstance(build, str):
        if build.isdigit():
            return int(build)

        m = re.search(r"/builds/(\d+)/?", build)
        if m:
            return int(m.group(1))

    return None


print("=" * 72)
print("CAPYBARA FRAMEWORK — READ THE DOCS BUILD")
print("=" * 72)

# ---------------------------------------------------------------------
# 1. Verify project
# ---------------------------------------------------------------------

_, project = rtd(
    "GET",
    f"projects/{RTD_PROJECT_SLUG}/"
)

print("\nProject:", project.get("name"))
print("Slug:   ", project.get("slug"))
print(
    "Repo:   ",
    project.get("repository", {}).get("url")
)
print("Branch: ", project.get("default_branch"))

# ---------------------------------------------------------------------
# 2. Sync versions first
# ---------------------------------------------------------------------

print("\nSynchronizing versions...")

status, _ = rtd(
    "POST",
    f"projects/{RTD_PROJECT_SLUG}/sync-versions/"
)

print("✓ Sync accepted:", status)

time.sleep(4)

# ---------------------------------------------------------------------
# 3. Verify latest
# ---------------------------------------------------------------------

_, version = rtd(
    "GET",
    f"projects/{RTD_PROJECT_SLUG}/versions/{RTD_VERSION}/"
)

print("\nVersion:")
print("  slug:      ", version.get("slug"))
print("  identifier:", version.get("identifier"))
print("  active:    ", version.get("active"))
print("  hidden:    ", version.get("hidden"))

if not version.get("active") or version.get("hidden"):
    print("\nActivating/unhiding latest...")

    rtd(
        "PATCH",
        f"projects/{RTD_PROJECT_SLUG}/versions/{RTD_VERSION}/",
        {
            "active": True,
            "hidden": False,
        }
    )

# ---------------------------------------------------------------------
# 4. Record existing build IDs
# ---------------------------------------------------------------------

_, existing = rtd(
    "GET",
    f"projects/{RTD_PROJECT_SLUG}/builds/?limit=20"
)

old_ids = {
    b.get("id")
    for b in existing.get("results", [])
    if b.get("id") is not None
}

# ---------------------------------------------------------------------
# 5. Trigger build
# ---------------------------------------------------------------------

print("\nTriggering fresh build...")

status, payload = rtd(
    "POST",
    f"projects/{RTD_PROJECT_SLUG}/versions/{RTD_VERSION}/builds/"
)

print("✓ Build accepted:", status)

build_id = get_build_id(payload)

# Some RTD responses don't expose the numeric build ID immediately.
if build_id is None:
    print("Waiting for new build to appear...")

    for _ in range(30):
        _, data = rtd(
            "GET",
            f"projects/{RTD_PROJECT_SLUG}/builds/?limit=20"
        )

        for b in data.get("results", []):
            bid = b.get("id")

            version_value = b.get("version")
            if isinstance(version_value, dict):
                version_value = version_value.get("slug")

            if (
                bid is not None
                and bid not in old_ids
                and version_value == RTD_VERSION
            ):
                build_id = bid
                break

        if build_id is not None:
            break

        time.sleep(2)

if build_id is None:
    raise RuntimeError(
        "RTD accepted the build request, but the new build ID "
        "could not be identified."
    )

print("✓ Build ID:", build_id)

dashboard_url = (
    f"https://app.readthedocs.org/projects/"
    f"{RTD_PROJECT_SLUG}/builds/{build_id}/"
)

print("Dashboard:", dashboard_url)

# ---------------------------------------------------------------------
# 6. Monitor build
# ---------------------------------------------------------------------

print("\nMonitoring build...")

last_state = None
build = None

for _ in range(180):

    _, build = rtd(
        "GET",
        f"projects/{RTD_PROJECT_SLUG}/builds/{build_id}/?expand=config"
    )

    state = build_state(build).lower()

    if state != last_state:
        print(
            time.strftime("%H:%M:%S"),
            "state =", state,
            "success =", build.get("success"),
        )
        last_state = state

    if state in {"finished", "cancelled"}:
        break

    time.sleep(5)

else:
    raise RuntimeError(
        f"Timed out monitoring build {build_id}.\n"
        f"Open: {dashboard_url}"
    )

# ---------------------------------------------------------------------
# 7. Result
# ---------------------------------------------------------------------

if not build.get("success"):

    print("\n✗ BUILD FAILED")
    print("Build ID:", build_id)
    print("State:   ", build_state(build))
    print("Commit:  ", build.get("commit"))
    print("Error:   ", build.get("error"))

    if globals().get("driver") is not None:
        driver.get(dashboard_url)

    raise RuntimeError(
        "Read the Docs build failed. "
        "The build dashboard was opened in Selenium if available."
    )

print("\n✓ BUILD SUCCEEDED")
print("Build ID:", build_id)
print("Commit:  ", build.get("commit"))
print("Duration:", build.get("duration"), "seconds")

# ---------------------------------------------------------------------
# 8. Show published URLs / offline formats
# ---------------------------------------------------------------------

_, version = rtd(
    "GET",
    f"projects/{RTD_PROJECT_SLUG}/versions/{RTD_VERSION}/"
)

docs_url = (
    version.get("urls", {}).get("documentation")
    or f"https://{RTD_PROJECT_SLUG}.readthedocs.io/en/{RTD_VERSION}/"
)

downloads = version.get("downloads") or {}

print("\nLIVE SITE")
print(docs_url)

print("\nREAD THE DOCS OFFLINE BUILDS")

for fmt in ("pdf", "epub", "htmlzip"):
    print(
        f"{fmt.upper():8}:",
        downloads.get(fmt, "(not reported)")
    )

if globals().get("driver") is not None:
    driver.get(docs_url)

print("\n" + "=" * 72)
print("CAPYBARA FRAMEWORK BUILD COMPLETE")
print("=" * 72)

Read the Docs API token (hidden):  ········


CAPYBARA FRAMEWORK — READ THE DOCS BUILD

Project: Capybara Framework
Slug:    capybara-framework
Repo:    https://github.com/BrianBowers-NapaCounty/itam-itsm_integration_framework
Branch:  main

Synchronizing versions...
✓ Sync accepted: 202

Version:
  slug:       latest
  identifier: main
  active:     True
  hidden:     False

Triggering fresh build...
✓ Build accepted: 202
✓ Build ID: 34542012
Dashboard: https://app.readthedocs.org/projects/capybara-framework/builds/34542012/

Monitoring build...
21:24:30 state = cloning success = None
21:25:03 state = building success = None
21:25:29 state = finished success = False

✗ BUILD FAILED
Build ID: 34542012
State:    finished
Commit:   fcd2316719f2eb55536c1df2138d8f4fa85abe41
Error:    


RuntimeError: Read the Docs build failed. The build dashboard was opened in Selenium if available.

In [5]:
# ================================================================
# CAPYBARA RC3 — REPAIR PRESERVATION AUDIT + PUSH + REBUILD RTD
# ================================================================

from pathlib import Path
import subprocess
import requests
import getpass
import time
import re
import json

ROOT = Path.cwd().resolve()

PROJECT = "capybara-framework"
VERSION = "latest"
RTD_API = "https://app.readthedocs.org/api/v3"

# ----------------------------------------------------------------
# REQUIRE REPOSITORY ROOT
# ----------------------------------------------------------------

required = [
    ROOT / ".git",
    ROOT / ".readthedocs.yaml",
    ROOT / "docs",
    ROOT / "tools" / "audit_rc3.py",
    ROOT / "tools" / "finalize_rc3.py",
    ROOT / "CONTRIBUTING.md",
]

missing = [str(p) for p in required if not p.exists()]

if missing:
    raise RuntimeError(
        "Run this cell from the Git repository root.\nMissing:\n  "
        + "\n  ".join(missing)
    )


def run(cmd, check=True):
    print("\n>", " ".join(map(str, cmd)))
    r = subprocess.run(
        [str(x) for x in cmd],
        cwd=ROOT,
        text=True,
        capture_output=True
    )

    if r.stdout.strip():
        print(r.stdout)

    if r.stderr.strip():
        print(r.stderr)

    if check and r.returncode != 0:
        raise RuntimeError(
            f"Command failed ({r.returncode}): {' '.join(map(str, cmd))}"
        )

    return r


# ----------------------------------------------------------------
# 1. FIX docs/contributing.md
#
# Preserve the EXACT substantive Alpha sentence that the audit
# protects, but materialize the root CONTRIBUTING.md directly.
# There is NO MyST {include}, so recursion cannot return.
# ----------------------------------------------------------------

legacy_notice = (
    "*This page mirrors the repository's "
    "*`/CONTRIBUTING.md`*. All changes should be made at the root.*"
)

canonical = (
    ROOT / "CONTRIBUTING.md"
).read_text(
    encoding="utf-8",
    errors="replace"
).strip()

docs_contrib = ROOT / "docs" / "contributing.md"

docs_contrib.write_text(
    legacy_notice
    + "\n\n"
    + canonical
    + "\n",
    encoding="utf-8"
)

print("✓ Repaired docs/contributing.md")
print("  - exact Alpha notice retained")
print("  - canonical content materialized")
print("  - recursive include removed")


# ----------------------------------------------------------------
# 2. PATCH FINALIZER SO IT KEEPS THIS FORM
# ----------------------------------------------------------------

finalizer = ROOT / "tools" / "finalize_rc3.py"

text = finalizer.read_text(
    encoding="utf-8",
    errors="replace"
)

old = '''            note = (
                f"This page mirrors the repository's `/{root_name}`. "
                "The documentation copy is materialized during release finalization "
                "to avoid recursive include behavior in MyST/Sphinx.\\n\\n"
            )
            destination.write_text(note + canonical + "\\n", encoding="utf-8")
'''

new = '''            if root_name == "CONTRIBUTING.md":
                note = (
                    "*This page mirrors the repository's "
                    "*`/CONTRIBUTING.md`*. All changes should be made at the root.*"
                    "\\n\\n"
                )
            else:
                note = (
                    f"This page mirrors the repository's `/{root_name}`. "
                    "The documentation copy is materialized during release finalization "
                    "to avoid recursive include behavior in MyST/Sphinx.\\n\\n"
                )

            destination.write_text(
                note + canonical + "\\n",
                encoding="utf-8"
            )
'''

if old in text:
    text = text.replace(old, new)

elif 'if root_name == "CONTRIBUTING.md":' not in text:
    raise RuntimeError(
        "Could not locate the expected finalizer mirror block. "
        "Stopping rather than modifying an unknown revision."
    )

finalizer.write_text(text, encoding="utf-8")

compile(
    finalizer.read_text(encoding="utf-8"),
    str(finalizer),
    "exec"
)

print("✓ Patched tools/finalize_rc3.py")


# ----------------------------------------------------------------
# 3. RECOVER ANY _static ASSETS REFERENCED BY THE DOCUMENTATION
#
# RC3 previously recovered only a few top-level _static files.
# Branding references additional logos/font specimens/etc.
# Pull any missing referenced static files from the still-live site.
# ----------------------------------------------------------------

STATIC_BASE = (
    "https://capybara-framework.readthedocs.io/en/latest/_static/"
)

static_pattern = re.compile(
    r'(?<![\w/])_static/([A-Za-z0-9_./()\- ]+?'
    r'\.(?:png|gif|jpg|jpeg|svg|ico|zip|woff|woff2|ttf|otf))',
    re.I
)

refs = set()

for ext in ("*.md", "*.rst"):
    for p in (ROOT / "docs").rglob(ext):
        if "_legacy_original_source" in p.parts:
            continue

        t = p.read_text(
            encoding="utf-8",
            errors="replace"
        )

        refs.update(static_pattern.findall(t))

session = requests.Session()
session.headers["User-Agent"] = "Capybara-RC3-static-repair/1.0"

recovered = []
not_found = []

for rel in sorted(refs):

    dest = ROOT / "docs" / "_static" / rel

    if dest.exists():
        continue

    url = STATIC_BASE + rel.replace("\\", "/")

    try:
        r = session.get(url, timeout=45)

        if r.ok and r.content:
            dest.parent.mkdir(
                parents=True,
                exist_ok=True
            )
            dest.write_bytes(r.content)

            recovered.append(rel)
            print("✓ recovered static:", rel)

        else:
            not_found.append(
                (rel, r.status_code)
            )

    except Exception as exc:
        not_found.append(
            (rel, str(exc))
        )

print(
    f"\nStatic recovery: "
    f"{len(recovered)} recovered; "
    f"{len(not_found)} unavailable."
)

if not_found:
    print("Unavailable static references:")
    for item in not_found:
        print("  ", item)


# ----------------------------------------------------------------
# 4. RUN THE STRICT PRESERVATION AUDIT
# ----------------------------------------------------------------

audit = run([
    "python",
    "tools/audit_rc3.py",
    "--strict"
], check=False)

if audit.returncode != 0:

    report_path = (
        ROOT
        / "legacy_baseline"
        / "RC3_COMPLETENESS_AUDIT.json"
    )

    if report_path.exists():
        report = json.loads(
            report_path.read_text(
                encoding="utf-8"
            )
        )

        print("\nPRESERVATION ERRORS:")
        for error in report.get("errors", []):
            print("  ERROR:", error)

    raise RuntimeError(
        "RC3 preservation audit still fails. "
        "Nothing has been pushed."
    )

print("\n✓ RC3 PRESERVATION AUDIT PASSED")


# ----------------------------------------------------------------
# 5. OPTIONAL LOCAL SPHINX TEST
# ----------------------------------------------------------------

try:
    import sphinx

    print("\nRunning local Sphinx HTML build...")

    local_build = run([
        "python",
        "-m",
        "sphinx",
        "-T",
        "-b",
        "html",
        "docs",
        "_build/rc3-test"
    ], check=False)

    if local_build.returncode != 0:
        raise RuntimeError(
            "Local Sphinx HTML build failed. "
            "Nothing has been pushed."
        )

    print("✓ Local Sphinx HTML build passed")

except ImportError:
    print(
        "\nSphinx is not installed in this notebook environment; "
        "skipping local Sphinx test."
    )


# ----------------------------------------------------------------
# 6. COMMIT + PUSH
# ----------------------------------------------------------------

run(["git", "add", "-A"])

status = run(
    ["git", "status", "--porcelain"],
    check=False
).stdout.strip()

if status:
    run([
        "git",
        "commit",
        "-m",
        "Repair RC3 preservation audit and missing static assets"
    ])
else:
    print("\nNo new Git changes to commit.")

confirm = input(
    "\nType PUSH RC3 FIX to push to origin/main: "
).strip()

if confirm != "PUSH RC3 FIX":
    raise RuntimeError("Push cancelled.")

run([
    "git",
    "push",
    "origin",
    "main"
])

print("\n✓ GitHub main updated")


# ----------------------------------------------------------------
# 7. READ THE DOCS API
# ----------------------------------------------------------------

RTD_API_TOKEN = globals().get(
    "RTD_API_TOKEN",
    ""
).strip()

if not RTD_API_TOKEN:
    RTD_API_TOKEN = getpass.getpass(
        "Read the Docs API token (hidden): "
    ).strip()

    globals()["RTD_API_TOKEN"] = RTD_API_TOKEN

if not RTD_API_TOKEN:
    raise RuntimeError(
        "No Read the Docs API token supplied."
    )


HEADERS = {
    "Authorization": f"Token {RTD_API_TOKEN}",
    "Accept": "application/json",
}


def api(method, path, body=None, ok=(200, 201, 202, 204)):

    url = (
        RTD_API.rstrip("/")
        + "/"
        + path.lstrip("/")
    )

    r = requests.request(
        method,
        url,
        headers=HEADERS,
        json=body,
        timeout=60
    )

    if r.status_code not in ok:
        raise RuntimeError(
            f"{method} {url}\n"
            f"HTTP {r.status_code}\n"
            f"{r.text[:3000]}"
        )

    try:
        return r.json()
    except Exception:
        return None


# Capture old builds so we can identify the new one.
existing = api(
    "GET",
    f"projects/{PROJECT}/builds/?limit=20"
)

old_ids = {
    x.get("id")
    for x in existing.get("results", [])
}


print("\nTriggering new Read the Docs build...")

trigger = api(
    "POST",
    f"projects/{PROJECT}/versions/{VERSION}/builds/"
)

build_id = None

if isinstance(trigger, dict):

    b = trigger.get("build")

    if isinstance(b, dict):
        build_id = b.get("id")

    elif isinstance(b, int):
        build_id = b


# If RTD didn't return the numeric ID immediately, discover it.
for _ in range(30):

    if build_id:
        break

    time.sleep(2)

    listing = api(
        "GET",
        f"projects/{PROJECT}/builds/?limit=20"
    )

    for item in listing.get("results", []):

        if item.get("id") not in old_ids:
            build_id = item.get("id")
            break


if not build_id:
    raise RuntimeError(
        "RTD accepted the build but I couldn't determine its build ID."
    )


dashboard = (
    f"https://app.readthedocs.org/projects/"
    f"{PROJECT}/builds/{build_id}/"
)

print("✓ RTD build ID:", build_id)
print("  Dashboard:", dashboard)


# ----------------------------------------------------------------
# 8. MONITOR
# ----------------------------------------------------------------

last = None
build = None

for _ in range(240):

    build = api(
        "GET",
        f"projects/{PROJECT}/builds/{build_id}/?expand=config"
    )

    state = build.get("state", "")

    if isinstance(state, dict):
        state = (
            state.get("code")
            or state.get("name")
            or str(state)
        )

    state = str(state).lower()

    if state != last:
        print(
            time.strftime("%H:%M:%S"),
            "RTD:",
            state
        )
        last = state

    if state in {"finished", "cancelled"}:
        break

    time.sleep(5)


# ----------------------------------------------------------------
# 9. RESULT
# ----------------------------------------------------------------

if not build.get("success"):

    print("\n" + "=" * 72)
    print("READ THE DOCS BUILD FAILED")
    print("=" * 72)

    print("Build ID:", build_id)
    print("Commit:  ", build.get("commit"))
    print("State:   ", build.get("state"))
    print("Error:")
    print(build.get("error"))

    print("\nDashboard:")
    print(dashboard)

    if globals().get("driver") is not None:
        driver.get(dashboard)

    raise RuntimeError(
        "RTD still failed; the detailed RTD error is printed immediately above."
    )


print("\n" + "=" * 72)
print("✓ CAPYBARA RC3 BUILD SUCCEEDED")
print("=" * 72)

print(
    f"https://{PROJECT}.readthedocs.io/en/{VERSION}/"
)

version = api(
    "GET",
    f"projects/{PROJECT}/versions/{VERSION}/"
)

downloads = version.get("downloads", {})

print("\nOffline formats:")
for fmt in ("pdf", "epub", "htmlzip"):
    print(
        f"  {fmt.upper():8}",
        downloads.get(fmt, "(not reported yet)")
    )

✓ Repaired docs/contributing.md
  - exact Alpha notice retained
  - canonical content materialized
  - recursive include removed


RuntimeError: Could not locate the expected finalizer mirror block. Stopping rather than modifying an unknown revision.

In [4]:
# ================================================================
# RC3 — FIX THE 3 LEGACY MIRROR NOTICES, AUDIT, COMMIT, PUSH
# ================================================================

from pathlib import Path
import subprocess
import json

ROOT = Path.cwd().resolve()

if not (ROOT / ".git").exists():
    raise RuntimeError("Run this from the Git repository root.")

def run(*args, check=True):
    print("\n>", " ".join(args))
    p = subprocess.run(
        list(args),
        cwd=ROOT,
        text=True,
        capture_output=True
    )
    if p.stdout.strip():
        print(p.stdout)
    if p.stderr.strip():
        print(p.stderr)
    if check and p.returncode:
        raise RuntimeError(
            f"Command failed ({p.returncode}): {' '.join(args)}"
        )
    return p


# ------------------------------------------------
# Exact notices from the live Alpha source
# ------------------------------------------------

MIRRORS = {
    "contributing.md": (
        "CONTRIBUTING.md",
        "*This page mirrors the repository's "
        "*`/CONTRIBUTING.md`*. All changes should be made at the root.*"
    ),
    "changelog.md": (
        "CHANGELOG.md",
        "*This page mirrors the repository's "
        "*`/CHANGELOG.md`*. All changes should be made at the root.*"
    ),
    "code-of-conduct.md": (
        "CODE_OF_CONDUCT.md",
        "*This page mirrors the repository's "
        "*`/CODE_OF_CONDUCT.md`*. All changes should be made at the root.*"
    ),
}


# ------------------------------------------------
# Rebuild docs mirrors WITHOUT {include}
# ------------------------------------------------

for docs_name, (root_name, exact_notice) in MIRRORS.items():

    src = ROOT / root_name
    dst = ROOT / "docs" / docs_name

    if not src.exists():
        raise RuntimeError(f"Missing canonical file: {src}")

    body = src.read_text(
        encoding="utf-8",
        errors="replace"
    ).strip()

    dst.write_text(
        exact_notice + "\n\n" + body + "\n",
        encoding="utf-8"
    )

    print(f"✓ {docs_name}")
    print("  legacy notice preserved exactly")
    print(f"  materialized from /{root_name}")


# ------------------------------------------------
# Patch finalizer so this stays fixed permanently
# ------------------------------------------------

finalizer = ROOT / "tools" / "finalize_rc3.py"

text = finalizer.read_text(
    encoding="utf-8",
    errors="replace"
)

# Replace the whole mirror-generation block from the previous hotfix.
start_marker = "    # Canonical mirrored repository documents flow ROOT -> docs only."
end_marker = "\n\ndef "

start = text.find(start_marker)

if start == -1:
    raise RuntimeError(
        "Expected RC3 mirror block not found in finalize_rc3.py."
    )

end = text.find(end_marker, start)

if end == -1:
    # Fallback: find next top-level function.
    import re
    m = re.search(r"\n(?=def\s+\w+\()", text[start:])
    if not m:
        raise RuntimeError(
            "Could not determine end of mirror block."
        )
    end = start + m.start()

replacement = r'''    # Canonical mirrored repository documents flow ROOT -> docs only.
    #
    # Preserve the exact Alpha-era mirror notices because they are part of
    # the protected legacy source. Materialize root content directly instead
    # of using MyST {include}; this prevents recursive parsing.
    mirror_docs = [
        (
            "CONTRIBUTING.md",
            "contributing.md",
            "*This page mirrors the repository's "
            "*`/CONTRIBUTING.md`*. All changes should be made at the root.*",
        ),
        (
            "CHANGELOG.md",
            "changelog.md",
            "*This page mirrors the repository's "
            "*`/CHANGELOG.md`*. All changes should be made at the root.*",
        ),
        (
            "CODE_OF_CONDUCT.md",
            "code-of-conduct.md",
            "*This page mirrors the repository's "
            "*`/CODE_OF_CONDUCT.md`*. All changes should be made at the root.*",
        ),
    ]

    for root_name, docs_name, legacy_notice in mirror_docs:
        source = repo / root_name
        destination = docs / docs_name

        if source.exists():
            canonical = source.read_text(
                encoding="utf-8",
                errors="replace"
            ).strip()

            destination.write_text(
                legacy_notice + "\n\n" + canonical + "\n",
                encoding="utf-8"
            )
'''

text = text[:start] + replacement + text[end:]

finalizer.write_text(text, encoding="utf-8")

# Make sure our edit is syntactically valid.
compile(
    finalizer.read_text(encoding="utf-8"),
    str(finalizer),
    "exec"
)

print("\n✓ finalize_rc3.py patched")


# ------------------------------------------------
# Remove stale wrapper files if present
# ------------------------------------------------

for rel in [
    "CONTRIBUTING.md.rst",
    "docs/contributing.md.rst",
    "docs/changelog.md.rst",
    "docs/code-of-conduct.md.rst",
]:
    p = ROOT / rel
    if p.exists():
        p.unlink()
        print("Removed:", rel)


# ------------------------------------------------
# Run STRICT audit
# ------------------------------------------------

audit = run(
    "python",
    "tools/audit_rc3.py",
    "--strict",
    check=False
)

report_path = (
    ROOT
    / "legacy_baseline"
    / "RC3_COMPLETENESS_AUDIT.json"
)

if report_path.exists():
    report = json.loads(
        report_path.read_text(encoding="utf-8")
    )

    print("\nAudit status:", report.get("status"))

    if report.get("warnings"):
        print("\nWarnings:")
        for x in report["warnings"]:
            print("  WARNING:", x)

    if report.get("errors"):
        print("\nErrors:")
        for x in report["errors"]:
            print("  ERROR:", x)

if audit.returncode != 0:
    raise RuntimeError(
        "Audit still fails. Copy the ERROR lines printed immediately "
        "above this message."
    )

print("\n" + "=" * 70)
print("✓ RC3 PRESERVATION AUDIT PASSED")
print("=" * 70)


# ------------------------------------------------
# Commit + push
# ------------------------------------------------

run("git", "add", "-A")

status = run(
    "git",
    "status",
    "--porcelain",
    check=False
).stdout.strip()

if status:
    run(
        "git",
        "commit",
        "-m",
        "Preserve Alpha mirror notices without recursive MyST includes"
    )
else:
    print("No new files need committing.")

confirm = input(
    "\nType PUSH to update GitHub main: "
).strip()

if confirm != "PUSH":
    raise RuntimeError("Push cancelled.")

run(
    "git",
    "push",
    "origin",
    "main"
)

print("\n✓ GitHub main updated.")
print("Now rerun your Read the Docs build cell.")

✓ contributing.md
  legacy notice preserved exactly
  materialized from /CONTRIBUTING.md
✓ changelog.md
  legacy notice preserved exactly
  materialized from /CHANGELOG.md
✓ code-of-conduct.md
  legacy notice preserved exactly
  materialized from /CODE_OF_CONDUCT.md

✓ finalize_rc3.py patched

> python tools/audit_rc3.py --strict
RC3 completeness audit: PASS — 0 error(s), 1 warning(s)


Audit status: PASS

Warnings:

✓ RC3 PRESERVATION AUDIT PASSED

> git add -A


> git status --porcelain
M  Capybara_RC3_Sphinx_Recursion_Hotfix.ipynb
A  docs/_static/assets/fonts/specimen_sheet-source_sans_3.png
M  docs/changelog.md
M  docs/code-of-conduct.md
M  docs/contributing.md
M  legacy_baseline/RC3_COMPLETENESS_AUDIT.json
M  tools/finalize_rc3.py


> git commit -m Preserve Alpha mirror notices without recursive MyST includes
[main 734ae9b] Preserve Alpha mirror notices without recursive MyST includes
 7 files changed, 1332 insertions(+), 36 deletions(-)
 create mode 100644 docs/_static/assets/fon


Type PUSH to update GitHub main:  PUSH



> git push origin main
To https://github.com/BrianBowers-NapaCounty/itam-itsm_integration_framework.git
   fcd2316..734ae9b  main -> main


✓ GitHub main updated.
Now rerun your Read the Docs build cell.


In [6]:
# =====================================================================
# CAPYBARA — TRIGGER + MONITOR READ THE DOCS BUILD
# =====================================================================

import requests
import getpass
import time
import re

RTD_PROJECT_SLUG = "capybara-framework"
RTD_VERSION = "latest"
RTD_API_BASE = "https://app.readthedocs.org/api/v3"

# Reuse an existing token from this kernel if available.
RTD_API_TOKEN = globals().get("RTD_API_TOKEN", "").strip()

if not RTD_API_TOKEN:
    RTD_API_TOKEN = getpass.getpass(
        "Read the Docs API token (hidden): "
    ).strip()
    globals()["RTD_API_TOKEN"] = RTD_API_TOKEN

if not RTD_API_TOKEN:
    raise RuntimeError("No Read the Docs API token supplied.")


def rtd(method, path, body=None, allowed=(200, 201, 202, 204)):
    url = (
        path
        if path.startswith("http")
        else RTD_API_BASE.rstrip("/") + "/" + path.lstrip("/")
    )

    headers = {
        "Accept": "application/json",
        "Authorization": f"Token {RTD_API_TOKEN}",
        "User-Agent": "Capybara-Framework-Build-Trigger/1.0",
    }

    if body is not None:
        headers["Content-Type"] = "application/json"

    response = requests.request(
        method.upper(),
        url,
        headers=headers,
        json=body,
        timeout=60,
    )

    try:
        data = response.json() if response.text else None
    except Exception:
        data = response.text

    if response.status_code not in allowed:
        raise RuntimeError({
            "method": method,
            "status": response.status_code,
            "url": url,
            "response": data,
        })

    return response.status_code, data


def build_state(build):
    state = build.get("state")

    if isinstance(state, dict):
        return (
            state.get("code")
            or state.get("name")
            or str(state)
        )

    return str(state or "")


def get_build_id(payload):
    if not isinstance(payload, dict):
        return None

    build = payload.get("build")

    if isinstance(build, int):
        return build

    if isinstance(build, dict):
        if build.get("id"):
            return int(build["id"])

        for key in ("url", "_links"):
            value = build.get(key)

            if isinstance(value, str):
                m = re.search(r"/builds/(\d+)/?", value)
                if m:
                    return int(m.group(1))

            elif isinstance(value, dict):
                for link in value.values():
                    if isinstance(link, str):
                        m = re.search(r"/builds/(\d+)/?", link)
                        if m:
                            return int(m.group(1))

    if isinstance(build, str):
        if build.isdigit():
            return int(build)

        m = re.search(r"/builds/(\d+)/?", build)
        if m:
            return int(m.group(1))

    return None


print("=" * 72)
print("CAPYBARA FRAMEWORK — READ THE DOCS BUILD")
print("=" * 72)

# ---------------------------------------------------------------------
# 1. Verify project
# ---------------------------------------------------------------------

_, project = rtd(
    "GET",
    f"projects/{RTD_PROJECT_SLUG}/"
)

print("\nProject:", project.get("name"))
print("Slug:   ", project.get("slug"))
print("Repo:   ", project.get("repository", {}).get("url"))
print("Branch: ", project.get("default_branch"))

# ---------------------------------------------------------------------
# 2. Sync versions first
# ---------------------------------------------------------------------

print("\nSynchronizing versions...")

status, _ = rtd(
    "POST",
    f"projects/{RTD_PROJECT_SLUG}/sync-versions/"
)

print("✓ Sync accepted:", status)

time.sleep(4)

# ---------------------------------------------------------------------
# 3. Verify latest
# ---------------------------------------------------------------------

_, version = rtd(
    "GET",
    f"projects/{RTD_PROJECT_SLUG}/versions/{RTD_VERSION}/"
)

print("\nVersion:")
print("  slug:      ", version.get("slug"))
print("  identifier:", version.get("identifier"))
print("  active:    ", version.get("active"))
print("  hidden:    ", version.get("hidden"))

if not version.get("active") or version.get("hidden"):
    print("\nActivating/unhiding latest...")

    rtd(
        "PATCH",
        f"projects/{RTD_PROJECT_SLUG}/versions/{RTD_VERSION}/",
        {
            "active": True,
            "hidden": False,
        }
    )

# ---------------------------------------------------------------------
# 4. Record existing build IDs
# ---------------------------------------------------------------------

_, existing = rtd(
    "GET",
    f"projects/{RTD_PROJECT_SLUG}/builds/?limit=20"
)

old_ids = {
    b.get("id")
    for b in existing.get("results", [])
    if b.get("id") is not None
}

# ---------------------------------------------------------------------
# 5. Trigger build
# ---------------------------------------------------------------------

print("\nTriggering fresh build...")

status, payload = rtd(
    "POST",
    f"projects/{RTD_PROJECT_SLUG}/versions/{RTD_VERSION}/builds/"
)

print("✓ Build accepted:", status)

build_id = get_build_id(payload)

if build_id is None:
    print("Waiting for new build to appear...")

    for _ in range(30):
        _, data = rtd(
            "GET",
            f"projects/{RTD_PROJECT_SLUG}/builds/?limit=20"
        )

        for b in data.get("results", []):
            bid = b.get("id")

            version_value = b.get("version")
            if isinstance(version_value, dict):
                version_value = version_value.get("slug")

            if (
                bid is not None
                and bid not in old_ids
                and version_value == RTD_VERSION
            ):
                build_id = bid
                break

        if build_id is not None:
            break

        time.sleep(2)

if build_id is None:
    raise RuntimeError(
        "RTD accepted the build request, but the new build ID "
        "could not be identified."
    )

print("✓ Build ID:", build_id)

dashboard_url = (
    f"https://app.readthedocs.org/projects/"
    f"{RTD_PROJECT_SLUG}/builds/{build_id}/"
)

print("Dashboard:", dashboard_url)

# ---------------------------------------------------------------------
# 6. Monitor build
# ---------------------------------------------------------------------

print("\nMonitoring build...")

last_state = None
build = None

for _ in range(180):

    _, build = rtd(
        "GET",
        f"projects/{RTD_PROJECT_SLUG}/builds/{build_id}/?expand=config"
    )

    state = build_state(build).lower()

    if state != last_state:
        print(
            time.strftime("%H:%M:%S"),
            "state =", state,
            "success =", build.get("success"),
        )
        last_state = state

    if state in {"finished", "cancelled"}:
        break

    time.sleep(5)

else:
    raise RuntimeError(
        f"Timed out monitoring build {build_id}.\n"
        f"Open: {dashboard_url}"
    )

# ---------------------------------------------------------------------
# 7. Result
# ---------------------------------------------------------------------

if not build.get("success"):

    print("\n" + "=" * 72)
    print("READ THE DOCS BUILD FAILED")
    print("=" * 72)

    print("Build ID:", build_id)
    print("Commit:  ", build.get("commit"))
    print("State:   ", build_state(build))
    print("Error:   ", build.get("error"))
    print("Dashboard:", dashboard_url)

    if globals().get("driver") is not None:
        driver.get(dashboard_url)

    raise RuntimeError(
        "Read the Docs build failed. "
        "Copy the Build ID / Commit / State / Error block printed above."
    )

print("\n" + "=" * 72)
print("✓ BUILD SUCCEEDED")
print("=" * 72)

print("Build ID:", build_id)
print("Commit:  ", build.get("commit"))
print("Duration:", build.get("duration"), "seconds")

# ---------------------------------------------------------------------
# 8. Published URLs / offline formats
# ---------------------------------------------------------------------

_, version = rtd(
    "GET",
    f"projects/{RTD_PROJECT_SLUG}/versions/{RTD_VERSION}/"
)

docs_url = (
    version.get("urls", {}).get("documentation")
    or f"https://{RTD_PROJECT_SLUG}.readthedocs.io/en/{RTD_VERSION}/"
)

downloads = version.get("downloads") or {}

print("\nLIVE SITE")
print(docs_url)

print("\nREAD THE DOCS OFFLINE BUILDS")

for fmt in ("pdf", "epub", "htmlzip"):
    print(
        f"{fmt.upper():8}:",
        downloads.get(fmt, "(not reported)")
    )

if globals().get("driver") is not None:
    driver.get(docs_url)

print("\n" + "=" * 72)
print("CAPYBARA FRAMEWORK BUILD COMPLETE")
print("=" * 72)

CAPYBARA FRAMEWORK — READ THE DOCS BUILD

Project: Capybara Framework
Slug:    capybara-framework
Repo:    https://github.com/BrianBowers-NapaCounty/itam-itsm_integration_framework
Branch:  main

Synchronizing versions...
✓ Sync accepted: 202

Version:
  slug:       latest
  identifier: main
  active:     True
  hidden:     False

Triggering fresh build...
✓ Build accepted: 202
✓ Build ID: 34542132
Dashboard: https://app.readthedocs.org/projects/capybara-framework/builds/34542132/

Monitoring build...
21:39:10 state = cloning success = None
21:39:21 state = installing success = None
21:39:42 state = building success = None
21:40:19 state = finished success = False

READ THE DOCS BUILD FAILED
Build ID: 34542132
Commit:   734ae9bd917eee7a26cb2ff7404d7b45bba6f8bd
State:    finished
Error:    
Dashboard: https://app.readthedocs.org/projects/capybara-framework/builds/34542132/


RuntimeError: Read the Docs build failed. Copy the Build ID / Commit / State / Error block printed above.

In [7]:
# =====================================================================
# CAPYBARA RC3 — REPAIR SPHINX/RTD + PUSH + REBUILD
# =====================================================================

from pathlib import Path
import subprocess
import re
import sys
import json
import time
import requests
import getpass
import importlib.util

ROOT = Path.cwd().resolve()
DOCS = ROOT / "docs"
LEGACY = DOCS / "_legacy_original_source"

PROJECT = "capybara-framework"
VERSION = "latest"
RTD_API = "https://app.readthedocs.org/api/v3"


# =====================================================================
# Helpers
# =====================================================================

required = [
    ROOT / ".git",
    ROOT / ".readthedocs.yaml",
    DOCS,
    DOCS / "conf.py",
    ROOT / "tools" / "audit_rc3.py",
]

missing = [str(p) for p in required if not p.exists()]

if missing:
    raise RuntimeError(
        "Run this cell from the Git repository root.\nMissing:\n  "
        + "\n  ".join(missing)
    )


def run(*args, check=True):
    print("\n>", " ".join(map(str, args)))

    p = subprocess.run(
        [str(x) for x in args],
        cwd=ROOT,
        text=True,
        capture_output=True,
    )

    if p.stdout.strip():
        print(p.stdout)

    if p.stderr.strip():
        print(p.stderr)

    if check and p.returncode:
        raise RuntimeError(
            f"Command failed ({p.returncode}): "
            + " ".join(map(str, args))
        )

    return p


print("=" * 72)
print("CAPYBARA RC3 — RTD REPAIR")
print("=" * 72)
print("Repository:", ROOT)


# =====================================================================
# 1. Make /LICENSE available for the preserved license page
# =====================================================================

license_md = ROOT / "LICENSE.md"
license_plain = ROOT / "LICENSE"

if license_md.exists() and not license_plain.exists():
    license_plain.write_text(
        license_md.read_text(encoding="utf-8", errors="replace"),
        encoding="utf-8",
    )
    print("✓ Created root LICENSE mirror from LICENSE.md")


# =====================================================================
# 2. Repair docs/license.md malformed literalinclude fence
# =====================================================================

license_page = DOCS / "license.md"

if license_page.exists():
    s = license_page.read_text(encoding="utf-8", errors="replace")

    # RC3 currently has a literalinclude fence without its closing ```
    marker = ":lines: 3-21\n\n<div"

    if marker in s:
        s = s.replace(
            marker,
            ":lines: 3-21\n```\n\n<div",
            1,
        )

        license_page.write_text(s, encoding="utf-8")
        print("✓ Closed malformed literalinclude fence in docs/license.md")


# =====================================================================
# 3. Restore or remove ONLY stale include directives whose targets
#    genuinely do not exist.
#
#    If the exact missing file exists in the recovered Alpha archive,
#    restore it. Otherwise the include itself is stale RC3 scaffolding;
#    remove the broken directive rather than publishing a CRITICAL error.
# =====================================================================

restored_includes = []
removed_stale_includes = []


def try_restore_legacy(target: Path):
    """Restore exact archived Alpha source when available."""

    try:
        rel = target.relative_to(DOCS)
    except ValueError:
        return False

    candidate = LEGACY / (str(rel).replace("\\", "/") + ".txt")

    if candidate.exists():
        target.parent.mkdir(parents=True, exist_ok=True)

        target.write_text(
            candidate.read_text(
                encoding="utf-8",
                errors="replace",
            ),
            encoding="utf-8",
        )

        restored_includes.append(str(rel))
        return True

    return False


# Markdown/MyST fenced includes
md_include = re.compile(
    r"(?ms)"
    r"^```\{include\}\s+([^\n]+)\n"
    r".*?"
    r"^```\s*$"
)

for p in DOCS.rglob("*.md"):

    if "_legacy_original_source" in p.parts:
        continue

    original = p.read_text(
        encoding="utf-8",
        errors="replace",
    )

    changed = False

    def fix_md_include(match):

        nonlocal_placeholder = None  # keeps replacement closure simple

        target_text = match.group(1).strip().strip("'\"")
        target = (p.parent / target_text).resolve()

        if target.exists():
            return match.group(0)

        if try_restore_legacy(target):
            return match.group(0)

        removed_stale_includes.append(
            f"{p.relative_to(ROOT)} -> {target_text}"
        )

        return (
            f"<!-- RC3: removed stale include for missing "
            f"non-legacy target: {target_text} -->"
        )

    repaired = md_include.sub(fix_md_include, original)

    if repaired != original:
        p.write_text(repaired, encoding="utf-8")


# RST includes
rst_include = re.compile(
    r"(?ms)"
    r"^\.\. include::\s+([^\n]+)\n"
    r"(?:^[ \t]+:[^\n]*\n|^[ \t]*\n)*"
)

for p in DOCS.rglob("*.rst"):

    if "_legacy_original_source" in p.parts:
        continue

    original = p.read_text(
        encoding="utf-8",
        errors="replace",
    )

    def fix_rst_include(match):

        target_text = match.group(1).strip().strip("'\"")
        target = (p.parent / target_text).resolve()

        if target.exists():
            return match.group(0)

        if try_restore_legacy(target):
            return match.group(0)

        removed_stale_includes.append(
            f"{p.relative_to(ROOT)} -> {target_text}"
        )

        return (
            ".. RC3: stale include removed because its target "
            f"does not exist in the recovered Alpha source: {target_text}\n\n"
        )

    repaired = rst_include.sub(fix_rst_include, original)

    if repaired != original:
        p.write_text(repaired, encoding="utf-8")


if restored_includes:
    print("\nRestored exact Alpha include targets:")
    for x in restored_includes:
        print("  ✓", x)

if removed_stale_includes:
    print("\nRemoved stale includes with no recoverable source:")
    for x in removed_stale_includes:
        print("  ✓", x)


# =====================================================================
# 4. Fix Alpha index's unclosed first toctree
#
#    Current source has:
#
#      ```{toctree}
#      ...
#      *
#
#      ---
#
#    so the second half of the page gets interpreted as toctree entries.
# =====================================================================

alpha_index = DOCS / "editions" / "alpha" / "index.md"

if alpha_index.exists():

    s = alpha_index.read_text(
        encoding="utf-8",
        errors="replace",
    )

    bad = ":titlesonly:\n*\n\n---"

    if bad in s:
        s = s.replace(
            bad,
            ":titlesonly:\n*\n```\n\n---",
            1,
        )

        alpha_index.write_text(
            s,
            encoding="utf-8",
        )

        print("✓ Closed malformed Alpha index toctree")


# =====================================================================
# 5. Fix malformed First Steps toctree
#
#    :hidden: is currently after an entry, so Sphinx thinks ':hidden:'
#    is another document.
# =====================================================================

first_steps = (
    DOCS
    / "editions"
    / "alpha"
    / "1-implementation-guide"
    / "1-1-first-steps"
    / "index.md"
)

if first_steps.exists():

    s = first_steps.read_text(
        encoding="utf-8",
        errors="replace",
    )

    bad = """:maxdepth: 2
:glob:
*
:hidden:
README
```"""

    good = """:maxdepth: 2
:glob:
:hidden:
*
```"""

    if bad in s:
        s = s.replace(bad, good, 1)

        first_steps.write_text(
            s,
            encoding="utf-8",
        )

        print("✓ Repaired First Steps toctree option order")


# =====================================================================
# 6. Remove empty :caption: options
# =====================================================================

for p in DOCS.rglob("*.md"):

    if "_legacy_original_source" in p.parts:
        continue

    s = p.read_text(
        encoding="utf-8",
        errors="replace",
    )

    repaired = re.sub(
        r"(?m)^:caption:\s*$\n?",
        "",
        s,
    )

    if repaired != s:
        p.write_text(repaired, encoding="utf-8")
        print("✓ Removed empty toctree caption:", p.relative_to(ROOT))


# =====================================================================
# 7. Fix nonexistent top-level "graphics" toctree target
# =====================================================================

root_index = DOCS / "index.md"

if root_index.exists():

    s = root_index.read_text(
        encoding="utf-8",
        errors="replace",
    )

    s2 = re.sub(
        r"(?m)^graphics\s*$",
        "editions/alpha/5-visual-resources/5-4-graphics",
        s,
    )

    if s2 != s:
        root_index.write_text(
            s2,
            encoding="utf-8",
        )

        print("✓ Repointed Supplemental Matter → Graphics catalog")


# =====================================================================
# 8. Remove illegal trailing document transition from docs/contributing
# =====================================================================

contrib = DOCS / "contributing.md"

if contrib.exists():

    s = contrib.read_text(
        encoding="utf-8",
        errors="replace",
    )

    # Preserve all actual content; only remove a terminal Markdown HR.
    s2 = re.sub(
        r"\n---\s*$",
        "\n",
        s,
    )

    if s2 != s:
        contrib.write_text(
            s2,
            encoding="utf-8",
        )

        print("✓ Removed terminal transition from docs/contributing.md")


# =====================================================================
# 9. Patch Sphinx 8.2.x LaTeX empty-table bug
#
# RTD is crashing here:
#
#   sphinx.builders.latex.transforms.LaTeXFootnoteVisitor.depart_table
#   tbody = next(node.findall(nodes.tbody))
#   StopIteration
#
# If a valid table contains only a header, Docutils may provide no tbody.
# We add an empty tbody before Sphinx's own transform runs.
# =====================================================================

conf = DOCS / "conf.py"

conf_text = conf.read_text(
    encoding="utf-8",
    errors="replace",
)

PATCH_MARKER = "CAPYBARA_RC3_LATEX_EMPTY_TABLE_GUARD"

if PATCH_MARKER not in conf_text:

    patch = r'''

# ---------------------------------------------------------------------
# CAPYBARA_RC3_LATEX_EMPTY_TABLE_GUARD
#
# Sphinx 8.2.x LaTeXFootnoteVisitor.depart_table() assumes every table
# contains a tbody and calls next(node.findall(nodes.tbody)) without a
# default. Header-only tables can legitimately lack tbody, causing
# StopIteration during RTD's PDF build.
#
# Preserve the table and supply an empty tbody before the built-in
# transform handles it.
# ---------------------------------------------------------------------

try:
    from docutils import nodes as _capybara_docutils_nodes
    from sphinx.builders.latex.transforms import (
        LaTeXFootnoteVisitor as _CapybaraLaTeXFootnoteVisitor,
    )

    _capybara_original_depart_table = (
        _CapybaraLaTeXFootnoteVisitor.depart_table
    )

    def _capybara_safe_depart_table(self, node):

        tbody = next(
            node.findall(_capybara_docutils_nodes.tbody),
            None,
        )

        if tbody is None:

            tgroup = next(
                node.findall(_capybara_docutils_nodes.tgroup),
                None,
            )

            if tgroup is not None:
                tgroup += _capybara_docutils_nodes.tbody()
            else:
                # No normal table structure exists. There is nowhere
                # appropriate to place table footnotes.
                self.table_footnotes = []
                return

        return _capybara_original_depart_table(self, node)

    _CapybaraLaTeXFootnoteVisitor.depart_table = (
        _capybara_safe_depart_table
    )

except Exception:
    # HTML/EPUB builds must not be affected if the LaTeX builder is
    # unavailable in another Sphinx environment.
    pass

'''

    conf_text += patch

    conf.write_text(
        conf_text,
        encoding="utf-8",
    )

    print("✓ Installed Sphinx LaTeX empty-table compatibility guard")
else:
    print("✓ LaTeX empty-table guard already installed")


# Ensure conf.py still compiles.
compile(
    conf.read_text(
        encoding="utf-8",
        errors="replace",
    ),
    str(conf),
    "exec",
)


# =====================================================================
# 10. STRICT RC3 PRESERVATION AUDIT
# =====================================================================

print("\n" + "=" * 72)
print("RUNNING STRICT PRESERVATION AUDIT")
print("=" * 72)

audit = run(
    sys.executable,
    "tools/audit_rc3.py",
    "--strict",
    check=False,
)

if audit.returncode != 0:

    report_path = (
        ROOT
        / "legacy_baseline"
        / "RC3_COMPLETENESS_AUDIT.json"
    )

    if report_path.exists():

        report = json.loads(
            report_path.read_text(
                encoding="utf-8",
                errors="replace",
            )
        )

        print("\nAUDIT ERRORS:")

        for error in report.get("errors", []):
            print("  ERROR:", error)

    raise RuntimeError(
        "Preservation audit failed. Nothing has been pushed."
    )

print("✓ RC3 preservation audit PASSED")


# =====================================================================
# 11. LOCAL SPHINX TEST — HTML + LATEX
#
# -b latex only generates the .tex tree. It does not require XeLaTeX
# itself, and specifically exercises the stage that crashed on RTD.
# =====================================================================

if importlib.util.find_spec("sphinx") is not None:

    print("\n" + "=" * 72)
    print("LOCAL SPHINX SMOKE TEST")
    print("=" * 72)

    html = run(
        sys.executable,
        "-m",
        "sphinx",
        "-T",
        "-b",
        "html",
        "docs",
        "_build/rc3-repair-html",
        check=False,
    )

    if html.returncode:
        raise RuntimeError(
            "Local HTML build failed. Nothing has been pushed."
        )

    print("✓ Local HTML build passed")

    latex = run(
        sys.executable,
        "-m",
        "sphinx",
        "-T",
        "-b",
        "latex",
        "docs",
        "_build/rc3-repair-latex",
        check=False,
    )

    if latex.returncode:
        raise RuntimeError(
            "Local LaTeX builder failed. Nothing has been pushed."
        )

    print("✓ Local LaTeX builder passed")

else:
    print(
        "\nSphinx is not installed in this kernel; "
        "local builds skipped. RTD will perform them."
    )


# =====================================================================
# 12. COMMIT + PUSH
# =====================================================================

run("git", "add", "-A")

status = run(
    "git",
    "status",
    "--porcelain",
    check=False,
).stdout.strip()

if status:

    print("\nChanges to commit:")
    print(status)

    run(
        "git",
        "commit",
        "-m",
        "Repair RC3 Sphinx structure and Read the Docs PDF build",
    )

else:
    print("\nNo Git changes remain to commit.")


confirm = input(
    "\nType PUSH RC3 to push the repaired source to GitHub: "
).strip()

if confirm != "PUSH RC3":
    raise RuntimeError("Push cancelled.")

run(
    "git",
    "push",
    "origin",
    "main",
)

print("\n✓ GitHub main updated")


# =====================================================================
# 13. READ THE DOCS — TRIGGER BUILD
# =====================================================================

RTD_API_TOKEN = globals().get(
    "RTD_API_TOKEN",
    "",
).strip()

if not RTD_API_TOKEN:

    RTD_API_TOKEN = getpass.getpass(
        "Read the Docs API token (hidden): "
    ).strip()

    globals()["RTD_API_TOKEN"] = RTD_API_TOKEN


if not RTD_API_TOKEN:
    raise RuntimeError("No Read the Docs token supplied.")


HEADERS = {
    "Authorization": f"Token {RTD_API_TOKEN}",
    "Accept": "application/json",
}


def api(method, path, body=None, ok=(200, 201, 202, 204)):

    url = (
        RTD_API.rstrip("/")
        + "/"
        + path.lstrip("/")
    )

    response = requests.request(
        method,
        url,
        headers=HEADERS,
        json=body,
        timeout=60,
    )

    if response.status_code not in ok:
        raise RuntimeError(
            f"{method} {url}\n"
            f"HTTP {response.status_code}\n"
            f"{response.text[:4000]}"
        )

    try:
        return response.json()
    except Exception:
        return None


existing = api(
    "GET",
    f"projects/{PROJECT}/builds/?limit=20",
)

old_ids = {
    x.get("id")
    for x in existing.get("results", [])
}


print("\nTriggering fresh Read the Docs build...")

trigger = api(
    "POST",
    f"projects/{PROJECT}/versions/{VERSION}/builds/",
)


# =====================================================================
# 14. Find new build ID
# =====================================================================

build_id = None

if isinstance(trigger, dict):

    b = trigger.get("build")

    if isinstance(b, int):
        build_id = b

    elif isinstance(b, dict):
        build_id = b.get("id")


for _ in range(30):

    if build_id:
        break

    time.sleep(2)

    listing = api(
        "GET",
        f"projects/{PROJECT}/builds/?limit=20",
    )

    for item in listing.get("results", []):

        bid = item.get("id")

        if bid not in old_ids:
            build_id = bid
            break


if not build_id:
    raise RuntimeError(
        "RTD accepted the build but the new build ID "
        "could not be determined."
    )


dashboard = (
    f"https://app.readthedocs.org/projects/"
    f"{PROJECT}/builds/{build_id}/"
)

raw_log = (
    f"https://app.readthedocs.org/api/v2/"
    f"build/{build_id}.txt"
)

print("✓ Build ID:", build_id)
print("Dashboard:", dashboard)


# =====================================================================
# 15. Monitor
# =====================================================================

last_state = None
build = None

for _ in range(240):

    build = api(
        "GET",
        f"projects/{PROJECT}/builds/{build_id}/?expand=config",
    )

    state = build.get("state", "")

    if isinstance(state, dict):
        state = (
            state.get("code")
            or state.get("name")
            or str(state)
        )

    state = str(state).lower()

    if state != last_state:
        print(
            time.strftime("%H:%M:%S"),
            "state =",
            state,
            "success =",
            build.get("success"),
        )

        last_state = state

    if state in {"finished", "cancelled"}:
        break

    time.sleep(5)


# =====================================================================
# 16. Result
# =====================================================================

if not build.get("success"):

    print("\n" + "=" * 72)
    print("READ THE DOCS BUILD FAILED")
    print("=" * 72)

    print("Build ID:", build_id)
    print("Commit:  ", build.get("commit"))
    print("State:   ", state)
    print("Error:   ", build.get("error"))
    print("Dashboard:", dashboard)
    print("Raw log:  ", raw_log)

    # Automatically print the tail of the public RTD raw log.
    try:
        log = requests.get(
            raw_log,
            timeout=60,
        ).text

        lines = log.splitlines()

        print("\nLAST 120 LOG LINES")
        print("-" * 72)

        print(
            "\n".join(lines[-120:])
        )

    except Exception as exc:
        print(
            "\nCould not retrieve raw log automatically:",
            exc,
        )

    raise RuntimeError(
        "RTD build still failed; the useful tail of the raw "
        "build log is printed immediately above."
    )


print("\n" + "=" * 72)
print("✓ CAPYBARA RC3 READ THE DOCS BUILD SUCCEEDED")
print("=" * 72)

print(
    f"https://{PROJECT}.readthedocs.io/en/{VERSION}/"
)

version = api(
    "GET",
    f"projects/{PROJECT}/versions/{VERSION}/",
)

downloads = version.get("downloads") or {}

print("\nOFFLINE FORMATS")

for fmt in ("pdf", "epub", "htmlzip"):
    print(
        f"{fmt.upper():8}:",
        downloads.get(fmt, "(not reported yet)")
    )

print("=" * 72)

CAPYBARA RC3 — RTD REPAIR
Repository: C:\Users\BBOWERS\Jupyter Notebooks\Capybara_Framework_GitHub_Repository_RC3_Integrated
✓ Created root LICENSE mirror from LICENSE.md
✓ Closed malformed literalinclude fence in docs/license.md

Removed stale includes with no recoverable source:
  ✓ docs\editions\alpha\1-implementation-guide.md -> 1-implementation-guide/README.md
  ✓ docs\editions\alpha\1-implementation-guide\1-2-indoors-setup\index.md -> README.md
  ✓ docs\editions\alpha\1-implementation-guide\1-3-integration\index.md -> README.md
  ✓ docs\editions\alpha\1-implementation-guide\1-4-utilization\index.md -> README.md
  ✓ docs\editions\alpha\1-implementation-guide\1-5-security-notes\index.md -> README.md
  ✓ docs\editions\alpha\4-scripts.rst -> 4-scripts/README.md
  ✓ docs\editions\alpha\6-document-templates.rst -> 6-document-templates/README.md
✓ Closed malformed Alpha index toctree
✓ Repaired First Steps toctree option order
✓ Removed empty toctree caption: docs\editions\alpha\0-overv

RuntimeError: Preservation audit failed. Nothing has been pushed.